# 05 — Untouched-model prompt and few-shot baselines

        **Estimated time:** 50 minutes<br>
        **Prerequisites:** 04 — Deterministic baselines<br>
        **Learner-produced evidence:** a validation comparison of basic, strong, and few-shot prompts

        ## Learning objectives

        - Hold model weights and validation examples constant across a prompt ladder.
- Build few-shot demonstrations exclusively from training records.
- Measure output quality and local resource use instead of eyeballing prose.

        This notebook is a teaching interface over the reusable code in `src/`.
        It uses only prepared local files. Run `make prepare-flight` before the
        trip; no cell installs packages or downloads data.


In [ ]:
from aai_local_finetuning.offline import enable_offline_environment

enable_offline_environment()

## Construct prompts before loading a model

The basic prompt states the task. The strong prompt adds allowed labels,
output shape, train-derived category mapping, escalation guidance, and
safety constraints. Few-shot adds deterministic demonstrations from train.
No prompt is selected using frozen-test failures.


In [ ]:
import pandas as pd

from aai_local_finetuning.evaluation import evaluate_predictions
from aai_local_finetuning.learning import (
    generate_support_predictions,
    load_support_splits,
    report_row,
    select_few_shots,
    support_contract,
)
from aai_local_finetuning.modeling import LocalMLXPredictor, build_messages
from aai_local_finetuning.settings import load_settings

settings = load_settings()
splits = load_support_splits(settings, include_test=False)
allowed_intents, categories = support_contract(splits.train)
demonstrations = select_few_shots(splits.train, limit=4)
validation_example = splits.validation[0]

## Inspect the controlled change

The final user message stays identical. Only the instruction/context
changes. Demonstration IDs are not needed in the prompt, but selection is
deterministic from training records and can be reconstructed.


In [ ]:
prompt_ladder = {
    strategy: build_messages(
        validation_example.input_text,
        strategy=strategy,
        allowed_intents=list(allowed_intents),
        category_by_intent=categories,
        few_shot=demonstrations,
    )
    for strategy in ("basic", "strong", "few_shot")
}
{
    strategy: {
        "message_count": len(messages),
        "system_preview": messages[0]["content"][:240],
        "same_final_user_message": (
            messages[-1]["content"] == validation_example.input_text
        ),
    }
    for strategy, messages in prompt_ladder.items()
}

## Small measured validation experiment

This default uses six validation examples so Run All remains practical on
a MacBook Air. It teaches mechanics, not statistical certainty. Increase
the limit only while prompts are still unlocked; record the final choice
before opening the frozen evaluation notebook.


In [ ]:
VALIDATION_LIMIT = 6
validation_probe = splits.validation[:VALIDATION_LIMIT]
predictor = LocalMLXPredictor(settings.model_dir)
prompt_reports = {}
prompt_predictions = {}
for strategy in ("basic", "strong", "few_shot"):
    predictions = generate_support_predictions(
        predictor,
        validation_probe,
        strategy=strategy,
        train_records=splits.train,
        max_tokens=96,
    )
    prompt_predictions[strategy] = predictions
    prompt_reports[strategy] = evaluate_predictions(
        validation_probe,
        predictions,
        supported_intents=allowed_intents,
    )
pd.DataFrame([report_row(name, report) for name, report in prompt_reports.items()])

## Inspect output as evidence, not as a vibe

A raw preview helps diagnose format errors. The strict report—not visual
plausibility—determines JSON parse, schema validity, supported labels,
classification, response policy, latency, tokens, and memory.


In [ ]:
[
    {
        "strategy": strategy,
        "example_id": predictions[0].example_id,
        "output_preview": predictions[0].raw_text[:300],
        "latency_ms": round(predictions[0].latency_ms, 1),
        "output_tokens": predictions[0].output_tokens,
    }
    for strategy, predictions in prompt_predictions.items()
]

## Exercise — lock a prompt strategy

Choose using validation evidence. Success means the rationale mentions
both classification and structured-output quality. Once notebook 07 is
opened, do not revise this choice in response to frozen-test errors.


In [ ]:
chosen_strategy = "strong"
choice_rationale = (
    "Use the constrained label and schema contract as the default; the "
    "small probe is insufficient to claim few-shot superiority."
)
assert chosen_strategy in prompt_reports
{
    "chosen_strategy": chosen_strategy,
    "validation_metrics": report_row(chosen_strategy, prompt_reports[chosen_strategy]),
    "rationale": choice_rationale,
}

**Hint:** prefer a reproducible rule over choosing the nicest single
output. A six-example probe can reject obvious failures, not establish a
final winner.


## Checkpoint

You have three untouched-model baselines and a validation-locked prompt
choice. The base weights have not changed.

**Next:** `06_lora_finetuning.ipynb` inspects and optionally runs the
adapter change without using frozen-test evidence.
